# Duka Electronics Sales Exploration

Half-year review of 1,000 orders from January through June 2026.

## Executive Summary

- The dataset contains **1,000 orders** from **January 1 through June 30, 2026**, with **0 missing values**.
- **Mombasa** led revenue with **KES 8,180,000**, while **Eldoret** led order count with **137 orders**; the rankings differed.
- **Laptop Pro 14** led revenue at **KES 16,840,500**, while **Bluetooth Speaker** led units sold with **162 units**; the top-three lists did not overlap.
- **698 weekday orders** generated **KES 37,119,000** and averaged **KES 53,179.08**, versus **302 weekend orders** averaging **KES 50,811.59**.
- **June** led at **KES 11,002,100**, **64.1%** above **February** at **KES 6,703,000**; the Premium Online Nairobi/Mombasa filter found **47 orders** worth **KES 4,891,100**.

## First-five-minutes inspection (Task 1)

The source file contains **1,000 rows and 9 columns**, with **0 missing values**. `order_date` parses as a date; order IDs, branches, regions, products, categories, and channels are text, while units and unit prices are integers. Dates run from **2026-01-01 through 2026-06-30**. The key categorical fields contain **8 branches, 5 regions, 12 products, 7 product categories, and 2 channels**. The following profile prints the full types, missing-value counts, categories, and first five records before any analysis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "plp-da-a3-duka-sales-exploration":
    PROJECT_DIR = Path("plp-da-a3-duka-sales-exploration").resolve()
DATA_PATH = PROJECT_DIR / "duka_sales.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run `python generate_data.py` from this project folder first.")

sales = pd.read_csv(DATA_PATH, parse_dates=["order_date"])
print(f"Loaded {len(sales):,} orders from {DATA_PATH.name}.")

Loaded 1,000 orders from duka_sales.csv.


In [2]:
print("Shape:", sales.shape)
print("\nColumn types:")
print(sales.dtypes.to_string())
print("\nMissing values:")
print(sales.isna().sum().to_string())
print("\nDate range:", sales["order_date"].min().date(), "to", sales["order_date"].max().date())
for column in ["branch", "region", "product", "category", "channel"]:
    print(f"\n{column} ({sales[column].nunique()} categories): {', '.join(sorted(sales[column].unique()))}")
display(sales.head())

Shape: (1000, 9)

Column types:
order_id                 str
order_date    datetime64[us]
branch                   str
region                   str
product                  str
category                 str
channel                  str
units_sold             int64
unit_price             int64

Missing values:
order_id      0
order_date    0
branch        0
region        0
product       0
category      0
channel       0
units_sold    0
unit_price    0

Date range: 2026-01-01 to 2026-06-30

branch (8 categories): Eldoret, Kisumu, Mombasa, Nairobi, Nakuru, Nyeri, Thika, Westlands

region (5 categories): Capital, Central, Coast, Rift Valley, Western

product (12 categories): 4K Television 43in, Bluetooth Speaker, Budget Smartphone, Chromebook Air, HD Television 32in, Laptop Pro 14, Power Bank 20K, Smart Watch, Smartphone X, Tablet 10in, Wi-Fi Router AX, Wireless Headphones

category (7 categories): Accessories, Audio, Computing, Mobile, Networking, Television, Wearables

channel (2 categori

,order_id,order_date,branch,region,product,category,channel,units_sold,unit_price
0,DUKA-0001,2026-03-24,Mombasa,Coast,Smart Watch,Wearables,Online,1,23100
1,DUKA-0002,2026-06-07,Nairobi,Capital,Laptop Pro 14,Computing,In-store,4,118900
2,DUKA-0003,2026-05-15,Nairobi,Capital,Smartphone X,Mobile,In-store,2,39700
3,DUKA-0004,2026-02-18,Thika,Central,HD Television 32in,Television,In-store,3,26600
4,DUKA-0005,2026-04-16,Kisumu,Western,Wireless Headphones,Audio,In-store,1,8400


## Derived columns (Task 2)

Revenue is units multiplied by unit price. Month and weekday come from the parsed order date; the weekend flag identifies Saturday and Sunday. The assignment defines Premium as a unit price of at least KES 28,000; lower prices are Value.

In [3]:
sales["revenue"] = sales["units_sold"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28_000, "Premium", "Value")

display(sales[["order_id", "order_date", "revenue", "month", "weekday", "is_weekend", "price_band"]].head())
assert len(sales) == 1_000
assert sales["revenue"].equals(sales["units_sold"] * sales["unit_price"])
assert (sales.loc[sales["price_band"] == "Premium", "unit_price"] >= 28_000).all()

,order_id,order_date,revenue,month,weekday,is_weekend,price_band
0,DUKA-0001,2026-03-24,23100,March,Tuesday,False,Value
1,DUKA-0002,2026-06-07,475600,June,Sunday,True,Premium
2,DUKA-0003,2026-05-15,79400,May,Friday,False,Premium
3,DUKA-0004,2026-02-18,79800,February,Wednesday,False,Value
4,DUKA-0005,2026-04-16,8400,April,Thursday,False,Value


## Task 3: Branch rankings

Rank every branch by total revenue and by order count. Compare positions directly; order count alone does not account for different product and order values.

In [4]:
branch_summary = (
    sales.groupby("branch", as_index=False)
    .agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
)
branch_by_revenue = branch_summary.sort_values(["revenue", "branch"], ascending=[False, True]).reset_index(drop=True)
branch_by_orders = branch_summary.sort_values(["orders", "branch"], ascending=[False, True]).reset_index(drop=True)
branch_by_revenue["revenue_rank"] = np.arange(1, len(branch_by_revenue) + 1)
branch_by_orders["orders_rank"] = np.arange(1, len(branch_by_orders) + 1)
branch_rankings = branch_by_revenue.merge(branch_by_orders[["branch", "orders_rank"]], on="branch")
branch_rankings["same_rank"] = branch_rankings["revenue_rank"] == branch_rankings["orders_rank"]
display(branch_rankings.sort_values("revenue_rank"))
print("Rankings identical:", branch_rankings["same_rank"].all())

,branch,revenue,orders,revenue_rank,orders_rank,same_rank
0,Mombasa,8180000,132,1,3,False
1,Nakuru,6860500,127,2,4,False
2,Nairobi,6775000,121,3,5,False
3,Kisumu,6671800,120,4,6,False
4,Westlands,6605900,118,5,7,False
5,Nyeri,6558700,136,6,2,False
6,Eldoret,5885500,137,7,1,False
7,Thika,4926700,109,8,8,True


Rankings identical: False


**Answer:** The rankings differ: **Mombasa** led revenue at **KES 8,180,000**, while **Eldoret** led order count with **137 orders**. Revenue also reflects product mix, units per order, and selling price, so order volume alone does not determine the revenue leader.

## Task 4: Product leaders

Rank products separately by total revenue and total units sold. A high-priced item can lead revenue even when another product sells more units.

In [5]:
product_summary = sales.groupby("product", as_index=False).agg(
    revenue=("revenue", "sum"), units=("units_sold", "sum")
)
top_revenue_products = product_summary.sort_values(["revenue", "product"], ascending=[False, True]).head(3)
top_units_products = product_summary.sort_values(["units", "product"], ascending=[False, True]).head(3)
print("Top 3 by revenue:")
display(top_revenue_products)
print("Top 3 by units sold:")
display(top_units_products)

Top 3 by revenue:


,product,revenue,units
5,Laptop Pro 14,16840500,145
0,4K Television 43in,6683000,124
8,Smartphone X,5952800,141


Top 3 by units sold:


,product,revenue,units
1,Bluetooth Speaker,1109900,162
3,Chromebook Air,4995000,157
6,Power Bank 20K,697300,155


**Answer:** The revenue leaders were **Laptop Pro 14**, **4K Television 43in**, and **Smartphone X**; the unit leaders were **Bluetooth Speaker**, **Chromebook Air**, and **Power Bank 20K**. The lists do not overlap: a revenue ranking weights units by price, while a unit ranking counts volume.

## Task 5: Weekend versus weekday orders

Compare order count, total revenue, and average order revenue. Average order revenue here means revenue per order, not revenue per calendar day.

In [6]:
weekend_summary = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean"),
)
weekend_summary.index = weekend_summary.index.map({False: "Weekday", True: "Weekend"})
display(weekend_summary.reindex(["Weekday", "Weekend"]).style.format({
    "total_revenue": "KES {:,.0f}", "average_order_revenue": "KES {:,.2f}"
}))

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,698,"KES 37,119,000","KES 53,179.08"
Weekend,302,"KES 15,345,100","KES 50,811.59"


**Answer:** Weekdays had **698 orders** worth **KES 37,119,000**, averaging **KES 53,179.08 per order**. Weekends had **302 orders** worth **KES 15,345,100**, averaging **KES 50,811.59 per order**. Weekday orders were more numerous and had a higher average order value.

## Task 6: Best and worst month

Aggregate revenue by calendar month, then compare the highest and lowest month. Percentage difference is calculated relative to the worst month's revenue.

In [7]:
MONTH_ORDER = ["January", "February", "March", "April", "May", "June"]
monthly_revenue = sales.groupby("month")["revenue"].sum().reindex(MONTH_ORDER)
display(monthly_revenue.rename("revenue_kes").to_frame().style.format("KES {:,.0f}"))
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
month_difference_pct = (monthly_revenue[best_month] - monthly_revenue[worst_month]) / monthly_revenue[worst_month] * 100
print(f"Best: {best_month} (KES {monthly_revenue[best_month]:,.0f})")
print(f"Worst: {worst_month} (KES {monthly_revenue[worst_month]:,.0f})")
print(f"Best month is {month_difference_pct:.1f}% above worst month.")

,revenue_kes
month,
January,"KES 8,121,500"
February,"KES 6,703,000"
March,"KES 7,645,500"
April,"KES 9,387,200"
May,"KES 9,604,800"
June,"KES 11,002,100"


Best: June (KES 11,002,100)
Worst: February (KES 6,703,000)
Best month is 64.1% above worst month.


**Answer:** **June** was the best month at **KES 11,002,100**; **February** was the worst at **KES 6,703,000**. June's revenue was **64.1%** higher than February's, using the worst month as the percentage baseline.

## Task 7: Premium Online orders in Nairobi and Mombasa

Filter to Premium orders (unit price ≥ KES 28,000), the Online channel, and the two requested branches. Save the matching source rows without an index.

In [8]:
premium_online_coast_capital = sales.loc[
    sales["price_band"].eq("Premium")
    & sales["channel"].eq("Online")
    & sales["branch"].isin(["Nairobi", "Mombasa"])
].copy()
output_dir = PROJECT_DIR / "outputs"
output_dir.mkdir(exist_ok=True)
export_path = output_dir / "premium_online_coast_capital.csv"
premium_online_coast_capital.to_csv(export_path, index=False)
print(f"Matching orders: {len(premium_online_coast_capital)}")
print(f"Total revenue: KES {premium_online_coast_capital['revenue'].sum():,.0f}")
print(f"Saved {export_path.relative_to(PROJECT_DIR)} without an index.")
display(premium_online_coast_capital.head())

Matching orders: 47
Total revenue: KES 4,891,100
Saved outputs\premium_online_coast_capital.csv without an index.


,order_id,order_date,branch,region,product,category,channel,units_sold,unit_price,revenue,month,weekday,is_weekend,price_band
29,DUKA-0030,2026-04-09,Nairobi,Capital,4K Television 43in,Television,Online,2,52600,105200,April,Thursday,False,Premium
78,DUKA-0079,2026-02-28,Mombasa,Coast,Smartphone X,Mobile,Online,1,41400,41400,February,Saturday,True,Premium
110,DUKA-0111,2026-06-19,Nairobi,Capital,Smartphone X,Mobile,Online,3,44600,133800,June,Friday,False,Premium
113,DUKA-0114,2026-04-19,Mombasa,Coast,4K Television 43in,Television,Online,1,51500,51500,April,Sunday,True,Premium
117,DUKA-0118,2026-04-02,Mombasa,Coast,Smartphone X,Mobile,Online,1,40200,40200,April,Thursday,False,Premium


**Answer:** The filter selected **47 orders** worth **KES 4,891,100**. This total is for complete order lines; it includes units sold multiplied by unit price, not simply the number of qualifying records.

## Task 8: Spreadsheet branch summary and PivotTable

The workbook contains the order-level data, a branch summary ranked by pandas with revenue calculated using `SUMIFS`, and a region lookup using `XLOOKUP`. The PivotTable summarizes revenue by branch and channel; Excel recalculates the workbook before its grand total is compared with the notebook total.

In [9]:
import re
import subprocess
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.worksheet.table import Table, TableStyleInfo

excel_dir = PROJECT_DIR / "excel"
excel_dir.mkdir(exist_ok=True)
workbook_path = excel_dir / "duka_branch_summary.xlsx"

orders_for_excel = sales.copy()
orders_for_excel["order_date"] = orders_for_excel["order_date"].dt.strftime("%Y-%m-%d")
workbook = Workbook()
orders_sheet = workbook.active
orders_sheet.title = "Orders"
orders_sheet.append(list(orders_for_excel.columns))
for row in orders_for_excel.itertuples(index=False, name=None):
    orders_sheet.append(list(row))
orders_table = Table(displayName="DukaOrders", ref=f"A1:{chr(64 + len(orders_for_excel.columns))}{len(orders_for_excel) + 1}")
orders_table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True, showColumnStripes=False)
orders_sheet.add_table(orders_table)
orders_sheet.freeze_panes = "A2"

region_sheet = workbook.create_sheet("Region Lookup")
region_sheet.append(["branch", "region"])
for branch, region in sorted(sales[["branch", "region"]].drop_duplicates().itertuples(index=False, name=None)):
    region_sheet.append([branch, region])

summary_sheet = workbook.create_sheet("Branch Summary")
summary_sheet.append(["branch", "revenue_kes", "region", "orders"])
for row_number, branch in enumerate(branch_by_revenue["branch"], start=2):
    summary_sheet.cell(row_number, 1, branch)
    summary_sheet.cell(row_number, 2, f"=SUMIFS(Orders!$J:$J,Orders!$C:$C,A{row_number})")
    summary_sheet.cell(row_number, 3, f'=IFERROR(XLOOKUP(A{row_number},\'Region Lookup\'!$A$2:$A$9,\'Region Lookup\'!$B$2:$B$9,"Not found"),VLOOKUP(A{row_number},\'Region Lookup\'!$A$2:$B$9,2,FALSE))')
    summary_sheet.cell(row_number, 4, f"=COUNTIF(Orders!$C:$C,A{row_number})")
summary_sheet.freeze_panes = "A2"

pivot_sheet = workbook.create_sheet("PivotTable")
pivot_sheet["A1"] = "Revenue by branch and channel"
pivot_sheet["A1"].font = Font(bold=True, size=14)
pivot_sheet["A2"] = "Generated as an Excel PivotTable from the Orders sheet when this notebook runs on Windows with Excel installed."
for sheet in workbook.worksheets:
    sheet.freeze_panes = sheet.freeze_panes or "A2"
    for cell in sheet[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1F4E78")
workbook.calculation.fullCalcOnLoad = True
workbook.calculation.forceFullCalc = True
workbook.calculation.calcMode = "auto"
workbook.save(workbook_path)

powershell_script = r'''$ErrorActionPreference = "Stop"
$excel = $null
$book = $null
try {
  $excel = New-Object -ComObject Excel.Application
  $excel.Visible = $false
  $excel.DisplayAlerts = $false
  $book = $excel.Workbooks.Open("WORKBOOK_PATH")
  $source = $book.Worksheets.Item("Orders")
  $pivotSheet = $book.Worksheets.Item("PivotTable")
  $sourceRange = $source.UsedRange
  $cache = $book.PivotCaches().Create(1, $sourceRange)
  $pivot = $cache.CreatePivotTable($pivotSheet.Range("A4"), "RevenueByBranchChannel")
  $branchField = $pivot.PivotFields("branch")
  $branchField.Orientation = 1
  $branchField.Position = 1
  $channelField = $pivot.PivotFields("channel")
  $channelField.Orientation = 2
  $channelField.Position = 1
  $revenueField = $pivot.AddDataField($pivot.PivotFields("revenue"), "Revenue (KES)", -4157)
  $revenueField.NumberFormat = "#,##0.00"
  $pivot.RowAxisLayout(1)
  $excel.CalculateFullRebuild()
  $book.Save()
  $grandTotal = $pivot.TableRange2.Cells.Item($pivot.TableRange2.Rows.Count, $pivot.TableRange2.Columns.Count).Value2
  Write-Output "PIVOT_TOTAL=$grandTotal"
} finally {
  if ($book -ne $null) { $book.Close($true) }
  if ($excel -ne $null) { $excel.Quit() }
  if ($pivot -ne $null) { [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($pivot) }
  if ($book -ne $null) { [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($book) }
  if ($excel -ne $null) { [void][System.Runtime.InteropServices.Marshal]::ReleaseComObject($excel) }
}
'''.replace("WORKBOOK_PATH", str(workbook_path).replace("'", "''"))

if __import__("os").name == "nt":
    completed = subprocess.run(
        ["powershell.exe", "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", powershell_script],
        check=True,
        capture_output=True,
        text=True,
    )
    pivot_match = re.search(r"PIVOT_TOTAL=([\d.]+)", completed.stdout)
    if not pivot_match:
        raise RuntimeError(f"Excel did not return a PivotTable total: {completed.stdout}")
    pivot_total = float(pivot_match.group(1))
    assert np.isclose(pivot_total, sales["revenue"].sum(), rtol=0, atol=0.01)
else:
    pivot_total = np.nan
    print("Excel PivotTable automation is available on Windows with desktop Excel.")

formula_book = load_workbook(workbook_path, data_only=False)
assert formula_book["Branch Summary"]["B2"].value.startswith("=SUMIFS(")
assert "XLOOKUP(" in formula_book["Branch Summary"]["C2"].value.upper()
formula_values = load_workbook(workbook_path, data_only=True)
assert formula_values["Branch Summary"]["C2"].value == sales.loc[sales["branch"].eq(branch_by_revenue.iloc[0]["branch"]), "region"].iloc[0]
assert len(formula_book["PivotTable"]._pivots) == 1 if __import__("os").name == "nt" else True
print(f"Workbook saved: {workbook_path.relative_to(PROJECT_DIR)}")
print(f"Pandas total revenue: KES {sales['revenue'].sum():,.2f}")
print(f"Excel PivotTable grand total: KES {pivot_total:,.2f}")
print("SUMIFS and XLOOKUP formulas verified; PivotTable total matches pandas.")

Workbook saved: excel\duka_branch_summary.xlsx
Pandas total revenue: KES 52,464,100.00
Excel PivotTable grand total: KES 52,464,100.00
SUMIFS and XLOOKUP formulas verified; PivotTable total matches pandas.


**Answer:** The branch summary uses `SUMIFS`, and the region lookup uses `XLOOKUP` with a `VLOOKUP` compatibility fallback for older Excel versions. The PivotTable groups revenue by branch and channel; its grand total and pandas total both equal **KES 52,464,100**, verified by the assertion in the preceding cell.